# 🧵 SareeDrape Studio — Full 150k GPU Training on Kaggle

**Model:** SareeTryOnModel (VITON-HD style, 113M params)  
**Runtime required:** GPU T4 × 2 or P100 — **DO NOT run on CPU**  
**Estimated time:** 4–8 hours · 20 epochs  
**Output:** `saree_tryon_model.pth` · `model_config.json` · `training_report.json` · `saree_weights.zip`

---

## ⚠️ BEFORE RUNNING — Complete ALL steps below first

### Step 1 — Enable GPU
```
Kaggle right panel → Settings → Accelerator → GPU T4 × 2
```
Confirm the kernel restarts. Cell 1 will raise an error if no GPU is found.

### Step 2 — Enable Internet
```
Kaggle right panel → Settings → Internet → On
```

### Step 3 — Add Kaggle API key as Secrets
```
Kaggle top menu → Add-ons → Secrets → + Add a new secret
  Name: KAGGLE_USERNAME   →  your Kaggle username
  Name: KAGGLE_KEY        →  your API key
```
Get API key: https://www.kaggle.com/settings → **API → Create New Token**

### Step 4 — Accept ALL dataset licences on Kaggle (one-time per account)
Open each link and click **Download** to accept the terms:

| # | Dataset | Purpose | Link |
|---|---|---|---|
| 1 | Saree Try-On Dataset | saree images | https://www.kaggle.com/datasets/chirag2466/saree-tryon-dataset |
| 2 | VITON-HD Zalando | body images + clothing | https://www.kaggle.com/datasets/marquis03/high-resolution-viton-zalando-dataset |
| 3 | Human Images (Women) | **women full-body with face** | https://www.kaggle.com/datasets/snmahsa/human-images-dataset-men-and-women |
| 4 | Indian Saree Patterns | **saree fabric / blouse material** | https://www.kaggle.com/datasets/div456/indian-saree-patterns |

### Step 5 — Run All Cells
```
Kaggle toolbar → ▶▶ Run All
```

### Step 6 — Download weights after Cell 11 completes
```
Kaggle right panel → Output tab → saree_weights.zip → ⬇ Download
```
Extract and place in your local project:
```
backend/dataset/trained_models/saree_tryon_model.pth
backend/dataset/trained_models/model_config.json
```
Restart Flask backend — it auto-detects the weights.

---

## Dataset mapping

| Kaggle Dataset | Slug | Images | → Training Category |
|---|---|---|---|
| Saree Try-On | `chirag2466/saree-tryon-dataset` | 33,564 | `saree_images` + `body_images` |
| VITON-HD Zalando | `marquis03/high-resolution-viton-zalando-dataset` | 109,432 | `body_images` (image/) + `blouse_materials` (cloth/) |
| **Human Images Women** | `snmahsa/human-images-dataset-men-and-women` | ~7,000 | **`body_images` (women folder only, full-body with face)** |
| **Indian Saree Patterns** | `div456/indian-saree-patterns` | 1,000+ | **`blouse_materials` (fabric: Ikat, Banarasi, Pichwai, Bandhani)** |

---

## Cell 1 — Environment check & install dependencies

Run this first. **If CUDA shows False, stop — enable GPU in Settings and restart kernel.**

In [ ]:
import subprocess, sys

pkgs = [
    'opencv-python-headless',
    'mediapipe',
    'tqdm',
    'Pillow',
    'numpy',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkg.split()])

import torch, platform
print(f'Python  : {platform.python_version()}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected! Go to Settings → Accelerator → GPU T4 × 2, "
        "then restart the kernel and run again."
    )

print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print('✅ Environment OK — proceed to Cell 2.')

## Cell 2 — Configuration

In [ ]:
from pathlib import Path
import json, os, torch

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/kaggle/working')
RAW_DIR     = BASE_DIR / 'dataset' / 'raw'
CLEANED_DIR = BASE_DIR / 'dataset' / 'cleaned'
ANNOT_DIR   = BASE_DIR / 'dataset' / 'annotations'
MODEL_DIR   = BASE_DIR / 'dataset' / 'trained_models'
CKPT_DIR    = MODEL_DIR / 'checkpoints'

for d in [RAW_DIR, CLEANED_DIR, ANNOT_DIR, MODEL_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

WEIGHTS_PATH = MODEL_DIR / 'saree_tryon_model.pth'
CONFIG_PATH  = MODEL_DIR / 'model_config.json'
REPORT_PATH  = MODEL_DIR / 'training_report.json'
ZIP_PATH     = BASE_DIR  / 'saree_weights.zip'

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_H        = 256
IMG_W        = 192
BATCH_SIZE   = 16      # T4 × 2 has enough VRAM for 16; reduce to 8 on P100
TOTAL_EPOCHS = 20
LR           = 1e-4
MAX_PER_CAT  = 50_000  # cap per category to keep training time manageable
GRID_SIZE    = 5
BASE_CH      = 64
CKPT_EVERY   = 2       # checkpoint every N epochs

DEVICE = torch.device('cuda')
print(f'Device      : {DEVICE} ({torch.cuda.get_device_name(0)})')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Epochs      : {TOTAL_EPOCHS}')
print(f'LR          : {LR}')
print(f'Image size  : {IMG_H}×{IMG_W}')
print(f'Max/cat     : {MAX_PER_CAT:,}')

## Cell 3 — Download all 4 datasets from Kaggle

### What gets downloaded and where it goes

| Dataset | Slug | Body images | Saree images | Blouse/fabric |
|---|---|---|---|---|
| Saree Try-On | `chirag2466/saree-tryon-dataset` | ✔ person folder | ✔ saree folder | — |
| VITON-HD | `marquis03/high-resolution-viton-zalando-dataset` | ✔ `train+test/image/` | — | ✔ `train+test/cloth/` |
| **Human Women** | `snmahsa/human-images-dataset-men-and-women` | ✔ **`women/` folder only** (full-body with face) | — | — |
| **Indian Saree Patterns** | `div456/indian-saree-patterns` | — | — | ✔ all images (Ikat/Banarasi/Pichwai/Bandhani fabric) |

Make sure you have accepted all dataset licences on Kaggle (see notebook header Step 4).

In [ ]:
import os, shutil
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

# ── Kaggle authentication ──────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = _s.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = _s.get_secret('KAGGLE_KEY')
    print('✅ Credentials loaded from Kaggle Secrets.')
except Exception as e:
    print(f'⚠️  Secrets unavailable ({e}). Falling back to ~/.kaggle/kaggle.json')

api = KaggleApi()
api.authenticate()
print('✅ Kaggle API authenticated.\n')

# ── Helpers ────────────────────────────────────────────────────────────────────
def count_imgs(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(1 for f in folder.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)

def copy_imgs(src: Path, dst: Path, limit: int = None, label: str = ''):
    """Flat-copy all images from src tree into dst (no subdirs)."""
    dst.mkdir(parents=True, exist_ok=True)
    files = [f for f in src.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS]
    if limit:
        files = files[:limit]
    copied = 0
    for f in files:
        target = dst / f.name
        if not target.exists():
            shutil.copy2(f, target)
            copied += 1
    tag = f'[{label}]' if label else ''
    print(f'  {tag} {src.relative_to(RAW_DIR) if RAW_DIR in src.parents else src.name}'
          f'  →  {dst.relative_to(BASE_DIR)}  ({copied:,} copied)')
    return copied

def download_ds(slug: str, dest: Path):
    dest.mkdir(parents=True, exist_ok=True)
    print(f'\n▶ Downloading  {slug}  →  {dest.name} ...')
    api.dataset_download_files(slug, path=str(dest), unzip=True, quiet=False)
    n = count_imgs(dest)
    print(f'  {n:,} images extracted total')
    return dest

# ══════════════════════════════════════════════════════════════════════════════
# DS1 — chirag2466/saree-tryon-dataset  (33,564 images)
# Typical structure: saree/ or saree_images/ and person/ or model/ or body/
# ══════════════════════════════════════════════════════════════════════════════
ds1 = download_ds('chirag2466/saree-tryon-dataset', RAW_DIR / 'ds1_saree_tryon')

SAREE_KW  = ['saree', 'cloth', 'garment', 'drape']
PERSON_KW = ['person', 'body', 'model', 'human', 'image', 'woman', 'women']

def best_subfolder(base: Path, keywords: list) -> Path:
    """Return subfolder matching keyword, or largest subfolder as fallback."""
    for kw in keywords:
        for d in sorted(base.rglob('*')):
            if d.is_dir() and kw.lower() in d.name.lower() and count_imgs(d) > 0:
                return d
    candidates = [d for d in base.rglob('*') if d.is_dir() and count_imgs(d) > 0]
    return max(candidates, key=count_imgs) if candidates else base

ds1_saree  = best_subfolder(ds1, SAREE_KW)
ds1_person = best_subfolder(ds1, PERSON_KW)
print(f'\n  DS1 saree  →  {ds1_saree.name}  ({count_imgs(ds1_saree):,} imgs)')
print(f'  DS1 person →  {ds1_person.name}  ({count_imgs(ds1_person):,} imgs)')

copy_imgs(ds1_saree,  RAW_DIR / 'saree_images',     label='DS1→saree')
copy_imgs(ds1_person, RAW_DIR / 'body_images',      label='DS1→body')

# ══════════════════════════════════════════════════════════════════════════════
# DS2 — marquis03/high-resolution-viton-zalando-dataset  (109,432 images)
# VITON-HD standard layout:
#   train/image/      person body photos   → body_images
#   train/cloth/      garment close-ups    → blouse_materials
#   test/image/       more person photos   → body_images
#   test/cloth/       more garments        → blouse_materials
# ══════════════════════════════════════════════════════════════════════════════
ds2 = download_ds('marquis03/high-resolution-viton-zalando-dataset', RAW_DIR / 'ds2_viton_hd')

for split in ['train', 'test']:
    person_src  = ds2 / split / 'image'
    garment_src = ds2 / split / 'cloth'
    if person_src.exists():
        copy_imgs(person_src,  RAW_DIR / 'body_images',      label=f'DS2-{split}→body')
    if garment_src.exists():
        copy_imgs(garment_src, RAW_DIR / 'blouse_materials', label=f'DS2-{split}→blouse')

# ══════════════════════════════════════════════════════════════════════════════
# DS3 — snmahsa/human-images-dataset-men-and-women  (~7,000 images)
# Structure:
#   women/    ← ONLY this folder — full-body women photos WITH FACE
#   men/      ← ignored (not relevant for saree draping)
# ══════════════════════════════════════════════════════════════════════════════
ds3 = download_ds('snmahsa/human-images-dataset-men-and-women', RAW_DIR / 'ds3_human_women')

# Find the women subfolder explicitly — skip men/
women_folder = None
for d in ds3.rglob('*'):
    if d.is_dir() and 'women' in d.name.lower() and 'men' not in d.name.lower().replace('women', ''):
        if count_imgs(d) > 0:
            women_folder = d
            break
# Fallback: folder named 'women' anywhere
if not women_folder:
    for d in ds3.rglob('*'):
        if d.is_dir() and d.name.lower() in ('women', 'woman', 'female'):
            if count_imgs(d) > 0:
                women_folder = d
                break

if women_folder:
    print(f'\n  DS3 women folder found: {women_folder.name}  ({count_imgs(women_folder):,} imgs)')
    copy_imgs(women_folder, RAW_DIR / 'body_images', label='DS3→body(women+face)')
else:
    print('  ⚠ DS3: "women" folder not found — skipping to avoid adding men images')

# ══════════════════════════════════════════════════════════════════════════════
# DS4 — div456/indian-saree-patterns  (1,000+ fabric images)
# Contains Indian fabric/textile patterns: Ikat, Banarasi, Pichwai, Bandhani
# These are close-up fabric/saree texture images → perfect for blouse_materials
# Structure: flat or class subfolders (Ikat/, Banarasi/, etc.)
# ══════════════════════════════════════════════════════════════════════════════
ds4 = download_ds('div456/indian-saree-patterns', RAW_DIR / 'ds4_saree_fabric')

# All images in this dataset are fabric textures — copy all to blouse_materials
copy_imgs(ds4, RAW_DIR / 'blouse_materials', label='DS4→blouse(fabric)')

# ── Final summary ──────────────────────────────────────────────────────────────
print('\n' + '='*65)
print('RAW DATASET SUMMARY')
print('='*65)
for cat in ['saree_images', 'body_images', 'blouse_materials']:
    n = count_imgs(RAW_DIR / cat)
    print(f'  {cat:<24}: {n:>8,} images')
total = sum(count_imgs(RAW_DIR / c) for c in ['saree_images', 'body_images', 'blouse_materials'])
print(f'  {"TOTAL":<24}: {total:>8,} images')
print('='*65)

## Cell 4 — Clean & resize images

Applies to all three categories. Each image is:
1. Rejected if unreadable or smaller than 64×64
2. Resized to 256×192 with Lanczos interpolation
3. CLAHE contrast normalisation in LAB colour space
4. Saved as PNG with light compression

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
MIN_DIM    = 64
CLAHE      = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

clean_report = {}

def clean_category(category, max_imgs=None):
    src = RAW_DIR     / category
    dst = CLEANED_DIR / category
    dst.mkdir(parents=True, exist_ok=True)

    paths = sorted(f for f in src.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTS)
    if max_imgs:
        paths = paths[:max_imgs]

    saved = rejected = 0
    for fpath in tqdm(paths, desc=f'  {category}', unit='img', ncols=80):
        img = cv2.imread(str(fpath))
        if img is None:
            rejected += 1
            continue
        h, w = img.shape[:2]
        if h < MIN_DIM or w < MIN_DIM:
            rejected += 1
            continue
        img = cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_LANCZOS4)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        img = cv2.cvtColor(cv2.merge([CLAHE.apply(l), a, b]), cv2.COLOR_LAB2BGR)
        cv2.imwrite(str(dst / (fpath.stem + '.png')), img, [cv2.IMWRITE_PNG_COMPRESSION, 1])
        saved += 1

    clean_report[category] = {'saved': saved, 'rejected': rejected, 'total_raw': saved + rejected}
    print(f'  {category}: {saved:,} cleaned  {rejected:,} rejected')
    return saved

print('Cleaning images...')
for cat in ['saree_images', 'body_images', 'blouse_materials']:
    clean_category(cat, max_imgs=MAX_PER_CAT)

print('\n' + '='*60)
print('CLEAN SUMMARY')
print('='*60)
for cat, r in clean_report.items():
    pct = 100 * r['saved'] / max(r['total_raw'], 1)
    print(f"  {cat:<22}: {r['saved']:>7,} saved  ({pct:.1f}% pass rate)")
print('='*60)

## Cell 5 — Generate pose annotations & segmentation masks

- **Body images** → MediaPipe 33-landmark pose keypoints → `annotations/<stem>_keypoints.json`
- **Saree + blouse images** → binary threshold mask → `cleaned/masks/<stem>_mask.png`

Note: MediaPipe on 50k+ images takes ~60–90 min on GPU (runs on CPU internally). This is normal.

In [ ]:
import json, cv2
import mediapipe as mp
from tqdm import tqdm

ANNOT_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR = CLEANED_DIR / 'masks'
MASK_DIR.mkdir(parents=True, exist_ok=True)

annot_report = {'body': {'processed': 0, 'failed': 0, 'total': 0}, 'masks': 0}
mp_pose = mp.solutions.pose

# ── Pose keypoints for body images ────────────────────────────────────────────
body_files = sorted((CLEANED_DIR / 'body_images').glob('*.png'))
annot_report['body']['total'] = len(body_files)
print(f'Annotating {len(body_files):,} body images with MediaPipe pose...')

with mp_pose.Pose(static_image_mode=True, model_complexity=1,
                  min_detection_confidence=0.3) as pose:
    for fpath in tqdm(body_files, desc='  Pose keypoints', unit='img', ncols=80):
        kp_path = ANNOT_DIR / (fpath.stem + '_keypoints.json')
        img = cv2.imread(str(fpath))
        if img is None:
            kp_path.write_text(json.dumps({'keypoints': [], 'error': 'read_failed'}))
            annot_report['body']['failed'] += 1
            continue
        try:
            result = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if result.pose_landmarks:
                kps = [
                    {'x': lm.x, 'y': lm.y, 'z': lm.z, 'visibility': lm.visibility}
                    for lm in result.pose_landmarks.landmark
                ]
                kp_path.write_text(json.dumps({'keypoints': kps, 'source': fpath.name}))
                annot_report['body']['processed'] += 1
            else:
                kp_path.write_text(json.dumps({'keypoints': [], 'note': 'no_pose_detected'}))
                annot_report['body']['failed'] += 1
        except Exception as e:
            kp_path.write_text(json.dumps({'keypoints': [], 'error': str(e)}))
            annot_report['body']['failed'] += 1

pose_rate = 100 * annot_report['body']['processed'] / max(len(body_files), 1)
print(f'  Pose detection: {annot_report["body"]["processed"]:,} OK  '
      f'{annot_report["body"]["failed"]:,} failed  ({pose_rate:.1f}% detection rate)')

# ── Binary masks for saree + blouse images ────────────────────────────────────
print('\nGenerating binary masks for saree & blouse images...')
for cat in ['saree_images', 'blouse_materials']:
    cat_files = sorted((CLEANED_DIR / cat).glob('*.png'))
    for fpath in tqdm(cat_files, desc=f'  Masks {cat}', unit='img', ncols=80):
        img = cv2.imread(str(fpath))
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)
        # Morphological clean-up
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        cv2.imwrite(str(MASK_DIR / (fpath.stem + '_mask.png')), mask)
        annot_report['masks'] += 1

# Save annotation report
(ANNOT_DIR / 'annotation_report.json').write_text(json.dumps(annot_report, indent=2))
print(f'\n  Masks saved: {annot_report["masks"]:,}')
print('  annotation_report.json saved.')

## Cell 6 — Model architecture (SareeTryOnModel)

Exact copy of `backend/ai_engine/ml_models.py` — must stay in sync with the local app.  
Runs a forward-pass sanity check at the end to confirm GPU is working.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, Dict

# ─────────────────────────────────────────────────────────────────────────────
# Building blocks
# ─────────────────────────────────────────────────────────────────────────────

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        return self.block(torch.cat([x, skip], dim=1))

class FeatureEncoder(nn.Module):
    def __init__(self, in_ch=3, base_ch=64):
        super().__init__()
        c = base_ch
        self.enc = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch, c,    4,2,1,bias=False), nn.LeakyReLU(0.2,True)),
            nn.Sequential(nn.Conv2d(c,     c*2,  4,2,1,bias=False), nn.InstanceNorm2d(c*2),  nn.LeakyReLU(0.2,True)),
            nn.Sequential(nn.Conv2d(c*2,   c*4,  4,2,1,bias=False), nn.InstanceNorm2d(c*4),  nn.LeakyReLU(0.2,True)),
            nn.Sequential(nn.Conv2d(c*4,   c*8,  4,2,1,bias=False), nn.InstanceNorm2d(c*8),  nn.LeakyReLU(0.2,True)),
        ])
    def forward(self, x):
        feats = []
        for layer in self.enc:
            x = layer(x); feats.append(x)
        return feats

class CorrelationLayer(nn.Module):
    def forward(self, fa, fb):
        B, C, H, W = fa.shape
        fa_flat = fa.view(B, C, H*W).permute(0,2,1)
        fb_flat = fb.view(B, C, H*W)
        corr    = torch.bmm(fa_flat, fb_flat) / (C**0.5)
        return corr.view(B, H*W, H, W)

class ThetaRegressor(nn.Module):
    _POOL_SIZE = 6
    def __init__(self, in_ch, grid_size=5):
        super().__init__()
        self.grid_size = grid_size
        self.pool = nn.AdaptiveAvgPool2d(self._POOL_SIZE)
        flat = in_ch * self._POOL_SIZE * self._POOL_SIZE
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, grid_size*grid_size*2), nn.Tanh(),
        )
    def forward(self, x):
        return self.net(self.pool(x)).view(-1, self.grid_size**2, 2)

class SareeWarpingNetwork(nn.Module):
    def __init__(self, grid_size=5, img_h=256, img_w=192, base_ch=64):
        super().__init__()
        self.grid_size   = grid_size
        self.img_h, self.img_w = img_h, img_w
        self.person_enc  = FeatureEncoder(in_ch=3+18, base_ch=base_ch)
        self.garment_enc = FeatureEncoder(in_ch=3,    base_ch=base_ch)
        self.corr        = CorrelationLayer()
        feat_h = img_h // 16; feat_w = img_w // 16
        self.theta = ThetaRegressor(feat_h * feat_w, grid_size)
    def _build_grid(self, theta, B, device):
        pts  = theta.view(B, self.grid_size, self.grid_size, 2)
        grid = F.interpolate(pts.permute(0,3,1,2).float(),
                             size=(self.img_h, self.img_w),
                             mode='bilinear', align_corners=True)
        return grid.permute(0,2,3,1)
    def forward(self, person_with_pose, garment, seg_mask=None):
        fp = self.person_enc(person_with_pose)[-1]
        fg = self.garment_enc(garment)[-1]
        if seg_mask is not None:
            sd = F.adaptive_avg_pool2d(seg_mask.float(), fp.shape[-2:])
            fp = fp * (0.5 + 0.5 * sd)
        corr   = self.corr(fp, fg)
        theta  = self.theta(corr)
        grid   = self._build_grid(theta, garment.size(0), garment.device)
        warped = F.grid_sample(garment, grid, mode='bilinear',
                               padding_mode='border', align_corners=True)
        return warped, grid

class TryOnGenerator(nn.Module):
    def __init__(self, in_ch=29, base_ch=64):
        super().__init__()
        c = base_ch
        self.d1 = DownBlock(in_ch, c,   normalize=False)
        self.d2 = DownBlock(c,     c*2)
        self.d3 = DownBlock(c*2,   c*4)
        self.d4 = DownBlock(c*4,   c*8, dropout=0.4)
        self.d5 = DownBlock(c*8,   c*8, dropout=0.4)
        self.d6 = DownBlock(c*8,   c*8, dropout=0.4)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(c*8, c*8, 3,1,1,bias=False), nn.InstanceNorm2d(c*8), nn.ReLU(True),
            nn.Conv2d(c*8, c*8, 3,1,1,bias=False), nn.InstanceNorm2d(c*8), nn.ReLU(True),
        )
        self.u1 = UpBlock(c*16, c*8, dropout=0.4)
        self.u2 = UpBlock(c*16, c*8, dropout=0.4)
        self.u3 = UpBlock(c*16, c*4)
        self.u4 = UpBlock(c*8,  c*2)
        self.u5 = UpBlock(c*4,  c)
        self.u6 = UpBlock(c*2,  c)
        self.out_head = nn.Sequential(nn.Conv2d(c, 4, 1), nn.Tanh())
    def forward(self, x):
        d1=self.d1(x); d2=self.d2(d1); d3=self.d3(d2)
        d4=self.d4(d3); d5=self.d5(d4); d6=self.d6(d5)
        bot=self.bottleneck(d6)
        u1=self.u1(bot,d6); u2=self.u2(u1,d5); u3=self.u3(u2,d4)
        u4=self.u4(u3,d3);  u5=self.u5(u4,d2); u6=self.u6(u5,d1)
        out=self.out_head(u6)
        rendered=out[:,:3]; alpha=(out[:,3:4]+1)/2
        composed=alpha*rendered+(1-alpha)*x[:,:3]
        return composed, alpha

class Discriminator(nn.Module):
    def __init__(self, in_ch=3, base_ch=64):
        super().__init__()
        c = base_ch
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, c,   4,2,1), nn.LeakyReLU(0.2,True),
            nn.Conv2d(c,    c*2,  4,2,1,bias=False), nn.InstanceNorm2d(c*2), nn.LeakyReLU(0.2,True),
            nn.Conv2d(c*2,  c*4,  4,2,1,bias=False), nn.InstanceNorm2d(c*4), nn.LeakyReLU(0.2,True),
            nn.Conv2d(c*4,  c*8,  4,1,1,bias=False), nn.InstanceNorm2d(c*8), nn.LeakyReLU(0.2,True),
            nn.Conv2d(c*8,  1,    4,1,1),
        )
    def forward(self, x):
        return self.net(x)

class SareeTryOnModel(nn.Module):
    N_STYLES = 16
    def __init__(self, img_h=256, img_w=192, grid_size=5, base_ch=64):
        super().__init__()
        self.img_h, self.img_w = img_h, img_w
        self.style_emb   = nn.Embedding(self.N_STYLES, 64)
        self.style_proj  = nn.Linear(64, img_h * img_w)
        self.saree_warp  = SareeWarpingNetwork(grid_size, img_h, img_w, base_ch)
        self.blouse_warp = SareeWarpingNetwork(grid_size, img_h, img_w, base_ch)
        self.generator   = TryOnGenerator(in_ch=29, base_ch=base_ch)
    def forward(self, person, saree, blouse, pose_heatmap, seg_mask, style_idx):
        style_e   = self.style_emb(style_idx.clamp(0, self.N_STYLES-1))
        style_map = self.style_proj(style_e).view(-1,1,self.img_h,self.img_w)
        pp = torch.cat([person, pose_heatmap], dim=1)
        ws, _  = self.saree_warp( pp, saree,  seg_mask)
        wb, _  = self.blouse_warp(pp, blouse, seg_mask)
        gen_in = torch.cat([person, ws, wb, pose_heatmap, seg_mask, style_map], dim=1)
        rendered, alpha = self.generator(gen_in)
        return {'rendered': rendered, 'alpha': alpha,
                'warped_saree': ws, 'warped_blouse': wb}

# Quick forward-pass test
model = SareeTryOnModel(img_h=IMG_H, img_w=IMG_W).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'Model params: {params/1e6:.2f}M')
with torch.no_grad():
    _t = lambda c: torch.randn(2,c,IMG_H,IMG_W, device=DEVICE)
    _out = model(person=_t(3), saree=_t(3), blouse=_t(3),
                 pose_heatmap=_t(18), seg_mask=_t(1),
                 style_idx=torch.zeros(2,dtype=torch.long,device=DEVICE))
print('Forward pass OK — rendered:', _out['rendered'].shape)

## Cell 7 — Dataset & DataLoader

Triplet dataset: each item is `(body_image, saree_image, blouse_image)`.  
Length = `min(len(body), len(saree), len(blouse))` for balanced sampling.

In [ ]:
import cv2, numpy as np
from torch.utils.data import Dataset, DataLoader

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

class SareeDataset(Dataset):
    """
    Triplet dataset: (body, saree, blouse).
    Balanced by minimum category size.
    """
    def __init__(self, cleaned_dir, img_h, img_w, max_per_cat=None):
        def _list(cat):
            p = sorted((cleaned_dir / cat).glob('*.png'))
            return p[:max_per_cat] if max_per_cat else p

        self.body   = _list('body_images')
        self.saree  = _list('saree_images')
        self.blouse = _list('blouse_materials')
        self.img_h  = img_h
        self.img_w  = img_w
        self.n      = min(len(self.body), len(self.saree), len(self.blouse))

        if self.n == 0:
            raise ValueError(
                'One or more categories have 0 images after cleaning! '
                'Check the dataset download and mapping in Cell 3.'
            )

        print(f'Dataset triplets:')
        print(f'  body_images      : {len(self.body):,}')
        print(f'  saree_images     : {len(self.saree):,}')
        print(f'  blouse_materials : {len(self.blouse):,}')
        print(f'  → Training on    : {self.n:,} triplets')

    def _load(self, fpath):
        img = cv2.imread(str(fpath))
        if img is None:
            img = np.zeros((self.img_h, self.img_w, 3), dtype=np.uint8)
        img = cv2.resize(img, (self.img_w, self.img_h), interpolation=cv2.INTER_LINEAR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        arr = img.astype(np.float32).transpose(2, 0, 1) / 127.5 - 1.0
        return torch.from_numpy(arr)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return (
            self._load(self.body  [idx % len(self.body)]),
            self._load(self.saree [idx % len(self.saree)]),
            self._load(self.blouse[idx % len(self.blouse)]),
        )

dataset = SareeDataset(CLEANED_DIR, IMG_H, IMG_W, max_per_cat=MAX_PER_CAT)

dataloader = DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = 4,
    pin_memory  = True,
    drop_last   = True,
    persistent_workers = True,
)
print(f'\nDataLoader: {len(dataloader):,} batches/epoch  (batch_size={BATCH_SIZE})')
print(f'Estimated steps total: {len(dataloader) * TOTAL_EPOCHS:,}')

## Cell 8 — Training loop

- **Optimiser:** Adam (β₁=0.5, β₂=0.999)
- **Scheduler:** CosineAnnealingLR
- **Loss:** MSE(rendered, person)
- **Grad clip:** 1.0
- Checkpoints every `CKPT_EVERY` epochs → `trained_models/checkpoints/`
- Best model always saved to `saree_tryon_model.pth`

In [ ]:
import time, datetime, torch, torch.nn as nn

optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.5, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6
)
criterion = nn.MSELoss()

best_loss  = float('inf')
train_log  = []
started_at = datetime.datetime.utcnow().isoformat()

print(f'Training started  : {started_at}')
print(f'Epochs            : {TOTAL_EPOCHS}')
print(f'Batches/epoch     : {len(dataloader):,}')
print(f'Device            : {DEVICE} ({torch.cuda.get_device_name(0)})')
print('=' * 70)

for epoch in range(1, TOTAL_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for step, (body, saree, blouse) in enumerate(dataloader):
        body   = body.to(DEVICE, non_blocking=True)
        saree  = saree.to(DEVICE, non_blocking=True)
        blouse = blouse.to(DEVICE, non_blocking=True)
        B      = body.size(0)

        pose_hm   = torch.zeros(B, 18, IMG_H, IMG_W, device=DEVICE)
        seg_mask  = torch.ones( B,  1, IMG_H, IMG_W, device=DEVICE)
        style_idx = torch.randint(0, SareeTryOnModel.N_STYLES, (B,), device=DEVICE)

        out  = model(person=body, saree=saree, blouse=blouse,
                     pose_heatmap=pose_hm, seg_mask=seg_mask,
                     style_idx=style_idx)
        loss = criterion(out['rendered'], body)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

        if step % 500 == 0:
            pct = 100 * step / len(dataloader)
            print(f'  e{epoch:02d} [{pct:5.1f}%] step {step:5d}/{len(dataloader)}  '
                  f'loss={loss.item():.6f}  '
                  f'gpu_mem={torch.cuda.memory_allocated()/1e9:.1f}GB')

    scheduler.step()

    avg_loss = epoch_loss / len(dataloader)
    elapsed  = time.time() - t0
    eta_h    = elapsed * (TOTAL_EPOCHS - epoch) / 3600
    lr_now   = scheduler.get_last_lr()[0]

    print(f'Epoch {epoch:02d}/{TOTAL_EPOCHS}  '
          f'avg_loss={avg_loss:.6f}  '
          f'lr={lr_now:.2e}  '
          f'time={elapsed/60:.1f}min  '
          f'ETA={eta_h:.1f}h')

    train_log.append({
        'epoch':     epoch,
        'loss':      round(avg_loss, 6),
        'lr':        lr_now,
        'elapsed_s': round(elapsed, 1),
    })

    # ── Checkpoint every N epochs ─────────────────────────────────────────────
    if epoch % CKPT_EVERY == 0:
        ckpt_path = CKPT_DIR / f'checkpoint_epoch{epoch:03d}.pth'
        torch.save({
            'epoch':           epoch,
            'model_state':     model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'loss':            round(avg_loss, 6),
            'stub':            False,
        }, str(ckpt_path))
        print(f'  ✔ Checkpoint saved → {ckpt_path.name}')

    # ── Save best model ───────────────────────────────────────────────────────
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch':         epoch,
            'model_state':   model.state_dict(),
            'loss':          round(best_loss, 6),
            'stub':          False,
            'is_prototype':  False,
            'training_mode': 'kaggle_full',
            'trained_at':    started_at,
            'total_imgs':    len(dataset),
            'device':        str(DEVICE),
            'img_h':         IMG_H,
            'img_w':         IMG_W,
            'datasets': [
                'chirag2466/saree-tryon-dataset',
                'marquis03/high-resolution-viton-zalando-dataset',
            ],
        }, str(WEIGHTS_PATH))
        print(f'  ✔ Best model saved  (loss={best_loss:.6f})')

print('=' * 70)
print(f'Training complete. Best loss: {best_loss:.6f}')

## Cell 9 — Save model_config.json & training_report.json

In [ ]:
import datetime, json

finished_at = datetime.datetime.utcnow().isoformat()

# ── model_config.json ─────────────────────────────────────────────────────────
config = {
    'img_h':         IMG_H,
    'img_w':         IMG_W,
    'grid_size':     GRID_SIZE,
    'base_ch':       BASE_CH,
    'epochs':        TOTAL_EPOCHS,
    'device':        str(DEVICE),
    'stub':          False,
    'is_prototype':  False,
    'training_mode': 'kaggle_full',
    'trained_at':    started_at,
    'finished_at':   finished_at,
    'total_imgs':    len(dataset),
    'best_loss':     round(best_loss, 6),
    'datasets': [
        {'slug': 'chirag2466/saree-tryon-dataset',                   'images': 33564},
        {'slug': 'marquis03/high-resolution-viton-zalando-dataset',  'images': 109432},
    ],
    'note': 'Full 150k Kaggle GPU training — SareeDrape Studio',
}
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print('✔ model_config.json saved')

# ── training_report.json ──────────────────────────────────────────────────────
size_mb = WEIGHTS_PATH.stat().st_size / 1024**2 if WEIGHTS_PATH.exists() else 0

report = {
    'training_mode':  'kaggle_full',
    'device':         str(DEVICE),
    'gpu':            torch.cuda.get_device_name(0),
    'total_epochs':   TOTAL_EPOCHS,
    'best_loss':      round(best_loss, 6),
    'started_at':     started_at,
    'finished_at':    finished_at,
    'dataset_sizes': {
        'body_images':      len(dataset.body),
        'saree_images':     len(dataset.saree),
        'blouse_materials': len(dataset.blouse),
    },
    'clean_report':   clean_report,
    'annot_report':   annot_report,
    'epoch_log':      train_log,
    'weights_size_mb': round(size_mb, 2),
    'model_params_m':  round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    'hyperparams': {
        'batch_size':  BATCH_SIZE,
        'lr':          LR,
        'img_h':       IMG_H,
        'img_w':       IMG_W,
        'max_per_cat': MAX_PER_CAT,
        'grid_size':   GRID_SIZE,
        'base_ch':     BASE_CH,
    },
    'source_datasets': [
        {'slug': 'chirag2466/saree-tryon-dataset',
         'url':  'https://www.kaggle.com/datasets/chirag2466/saree-tryon-dataset',
         'images': 33564, 'category_mapping': 'saree_images + body_images'},
        {'slug': 'marquis03/high-resolution-viton-zalando-dataset',
         'url':  'https://www.kaggle.com/datasets/marquis03/high-resolution-viton-zalando-dataset',
         'images': 109432, 'category_mapping': 'body_images (image/) + blouse_materials (cloth/)'},
    ],
}
REPORT_PATH.write_text(json.dumps(report, indent=2))
print('✔ training_report.json saved')

print(f'\nOutput files:')
print(f'  {WEIGHTS_PATH.name:<35} {size_mb:.1f} MB')
print(f'  {CONFIG_PATH.name}')
print(f'  {REPORT_PATH.name}')

## Cell 10 — Verify weights (sanity check)

Reloads the saved checkpoint and runs a forward pass to confirm:
- `stub = False` ✔
- `is_prototype = False` ✔
- `model_state` loads cleanly ✔
- Forward pass produces correct output shape ✔

In [ ]:
import torch

print('Verifying saved weights...')
ckpt = torch.load(str(WEIGHTS_PATH), map_location='cpu')

assert not ckpt.get('stub', True),         '✗ stub flag must be False'
assert not ckpt.get('is_prototype', True), '✗ is_prototype must be False'
assert 'model_state' in ckpt,             '✗ model_state key missing'

verify_model = SareeTryOnModel(img_h=IMG_H, img_w=IMG_W)
verify_model.load_state_dict(ckpt['model_state'])
verify_model.eval()

with torch.no_grad():
    _t = lambda c: torch.randn(1, c, IMG_H, IMG_W)
    out = verify_model(
        person       = _t(3),
        saree        = _t(3),
        blouse       = _t(3),
        pose_heatmap = _t(18),
        seg_mask     = _t(1),
        style_idx    = torch.tensor([0]),
    )

print(f'✔ rendered shape   : {out["rendered"].shape}  (expected [1, 3, {IMG_H}, {IMG_W}])')
print(f'✔ alpha shape      : {out["alpha"].shape}')
print(f'✔ epoch trained    : {ckpt["epoch"]}')
print(f'✔ best loss        : {ckpt["loss"]}')
print(f'✔ stub             : {ckpt.get("stub")}')
print(f'✔ is_prototype     : {ckpt.get("is_prototype")}')
print(f'✔ training_mode    : {ckpt.get("training_mode")}')
print(f'✔ datasets         : {ckpt.get("datasets")}')
print('\n✅ Weights verified — proceed to Cell 11 to zip and download.')

## Cell 11 — Zip outputs & prepare download

Creates `saree_weights.zip` in `/kaggle/working/` containing:
- `saree_tryon_model.pth` — trained weights
- `model_config.json` — hyperparameters & metadata
- `training_report.json` — full training log

**After this cell:** go to the Kaggle **Output** panel → click `saree_weights.zip` → **Download**.

In [ ]:
import zipfile

files_to_zip = [
    (WEIGHTS_PATH, 'saree_tryon_model.pth'),
    (CONFIG_PATH,  'model_config.json'),
    (REPORT_PATH,  'training_report.json'),
]

# Also include all epoch checkpoints
for ckpt_file in sorted(CKPT_DIR.glob('checkpoint_epoch*.pth')):
    files_to_zip.append((ckpt_file, f'checkpoints/{ckpt_file.name}'))

print(f'Creating {ZIP_PATH.name} ...')
with zipfile.ZipFile(str(ZIP_PATH), 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for src, arcname in files_to_zip:
        if src.exists():
            zf.write(str(src), arcname)
            size_mb = src.stat().st_size / 1024**2
            print(f'  + {arcname:<45} {size_mb:.1f} MB')
        else:
            print(f'  ⚠ MISSING: {arcname}')

zip_size_mb = ZIP_PATH.stat().st_size / 1024**2
print(f'\n✅ {ZIP_PATH.name}  →  {zip_size_mb:.1f} MB total')
print()
print('━'*60)
print('DOWNLOAD INSTRUCTIONS')
print('━'*60)
print('1. Kaggle right panel → Output tab')
print(f'2. Click  saree_weights.zip')
print('3. Click ⬇ Download')
print()
print('After downloading, extract and place in your local project:')
print()
print('  backend/dataset/trained_models/saree_tryon_model.pth')
print('  backend/dataset/trained_models/model_config.json')
print()
print('Then restart Flask backend (python app.py)')
print('The engine will auto-load the weights on startup.')
print('━'*60)